In [23]:
# Importing torch
import torch
import torchvision
import torchmetrics # torch, torchvision, torchaudio and torchmetrics must be downloaded on the same GPU compatability → cu126

import torchsummary
import torchinfo

print(torch.__version__, torchvision.__version__, torchmetrics.__version__, torchinfo.__version__, sep="\n")
print(torchsummary.__name__) # does not have __version__ method

2.14.0+cu126
0.29.0+cu126
1.9.0
1.8.0
torchsummary


In [24]:
# GPU avaliability
if torch.cuda.is_available():
    print("GPU is available!")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"cuda version: {torch.version.cuda}")
else:
    print("GPU not available. Using CPU.")

!nvidia-smi # cuda configurations

GPU is available!
Using GPU: NVIDIA GeForce MX350
cuda version: 12.6
Tue Sep 22 10:50:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce MX350           Off |   00000000:01:00.0 Off |                  N/A |
| N/A   45C    P8            N/A  / 5001W |      54MiB /   2048MiB |      0%      Default |
|                                         |                        |   

---

In [25]:
# This method will assign a memory to create a tensor of provided shape and return the garbage/existing numbers at that location
a: torch.Tensor = torch.empty(
    size=[2, 3],
    dtype=torch.float32,
    out=torch.ones(size=[2, 3], dtype=torch.float32, device="cuda"), # shape, dtype & device of both the tensors must be same
    device="cuda",
    pin_memory=True,
    requires_grad=False # operations that use the out parameter are generally non-differentiable in terms of the autograd graph (i.e., they break the computation graph and will typically throw an error if requires_grad=True is involved or cause issues during backpropagation).
)

print(a)

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')


The `out` parameter allows you to specify a pre-allocated tensor where the result of an operation will be written, rather than creating a brand new tensor in memory. <br>

It prevents PyTorch from allocating new memory for the output. This is especially useful in loops, continuous inference pipelines, or memory-constrained environments to avoid garbage collection overhead.

Scaler (0D) → Vector (1D) → Tensor (> 1D)

In [26]:
# Check type
type(a)

torch.Tensor

In [27]:
# Check the memory consuption
from sys import getsizeof

print(f"Total Memory {getsizeof(a)}")
print(f"Tensor Memory {a.nbytes}") # Float of 4 bytes

# Total Memory → 6(total numbers) * [4(size of float32) + 8(size of pointer)] = 72

Total Memory 72
Tensor Memory 24


The core difference is that `sys.getsizeof()` measures the total memory footprint of an object (including overhead and references), while `.nbytes` measures only the raw data size of the object's elements, excluding overhead.

In [28]:
# Using zeros
torch.zeros(
    size = [2, 3], 
    dtype = torch.float32, 
    device="cpu",
    layout=torch.strided, # how a tensor's data is structured and stored in memory. For dense tensors → torch.strided & for sparse tensors → torch.sparse_csr
    pin_memory=True, 
    requires_grad=False
)

tensor([[0., 0., 0.],
        [0., 0., 0.]])

In [29]:
# using ones
torch.ones(size = [2, 3])

tensor([[1., 1., 1.],
        [1., 1., 1.]])

In [30]:
# using rand
torch.rand(size = [2, 3])

tensor([[0.1367, 0.0473, 0.2435],
        [0.5084, 0.1512, 0.6271]])

In [31]:
# manual_seed - for regenerating the same array
seed_number: torch.long = torch.initial_seed() # `torch.random.seed()` store this seed number and use it as a variable → python long datatype
print(seed_number)
# NOTE: Seed number act as a starting point to generate the sequences and numbers available on those locations will be in the resultant tensor

torch.manual_seed(seed=seed_number) # `torch.random.manual_seed(seed_number)`
torch.rand(2,3) # in notebooks, regenerating with seed number limited to this cell only → store it in .json files and use it throughout your project

220919085250671801


tensor([[0.3443, 0.8929, 0.3529],
        [0.1352, 0.2100, 0.4003]])

In [32]:
# Device agnostic code
from typing import Literal
device: Literal['cpu', 'cuda'] = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [33]:
# using tensor
tensor: torch.Tensor = torch.tensor(
    data = [[1,2,3],[4,5,6]], # list or tuple
    dtype = torch.float32,
    requires_grad = False,
    device = "cpu",
    pin_memory = True # only if tensor is on CPU and want to transfer it on GPU later
); tensor.to(device=device); print(tensor)

# from numpy arrays
import numpy as np

array: np.ndarray = np.array(object=(1, 2, 3))
tensor_from_numpy: torch.Tensor = torch.from_numpy(array)
print(tensor_from_numpy)

tensor([[1., 2., 3.],
        [4., 5., 6.]])
tensor([1, 2, 3])


Mentioning `device → cuda` as well as `pin_memory → True` is a bug

In [36]:
# Question: Which one is fast?
torch.rand(size=[2, 3]).to(device="cuda") # Slow: CPU (pager memory) → RAM (page-locaked memory) → VRAM(GPU memory)
torch.rand(size=[2, 3], pin_memory=True).to(device="cuda") # Itermediate: RAM (page-locaked memory) → VRAM(GPU memory)
torch.rand(size=[2, 3], device="cuda") # Fast: VRAM(GPU memory)

tensor([[0.9734, 0.3550, 0.8814],
        [0.7275, 0.2188, 0.7403]], device='cuda:0')

[A guide on good usage of non_blocking and pin_memory() in PyTorch](https://docs.pytorch.org/tutorials/intermediate/pinmem_nonblock.html)

In [37]:
# NOTE: You can only calculate the gradient when the end result (loss function) is scaler.
try:
    a = torch.rand(size=[2, 3], requires_grad=True, pin_memory=True).to(device=device)
    b: torch.Tensor = a.pow(exponent=2)
    c: torch.Tensor = b * 2 # c = (a ^ 2) + 2

    c.backward()

except RuntimeError as e:
    print(e)

grad can be implicitly created only for scalar outputs


In [40]:
# arange
print("using arange ->", torch.arange(start=0, end=10, step=2))

# using linspace
print("using linspace ->", torch.linspace(start=0, end=10, steps=10))

# using eye - Identity Matrix tensor
print("using eye ->", torch.eye(n=5))

# using full - torch.ones(size = (m, n)) * constant( = 5)
print("using full ->", torch.full(size=(3, 3), fill_value=5))
print(f"using fill -> {torch.fill(input=torch.ones(size=[2, 3]), value=3)}") # replace all the numbers with constant value

using arange -> tensor([0, 2, 4, 6, 8])
using linspace -> tensor([ 0.0000,  1.1111,  2.2222,  3.3333,  4.4444,  5.5556,  6.6667,  7.7778,
         8.8889, 10.0000])
using eye -> tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]])
using full -> tensor([[5, 5, 5],
        [5, 5, 5],
        [5, 5, 5]])
using fill -> tensor([[3., 3., 3.],
        [3., 3., 3.]])


In [ ]:
torch.logspace(start=0, end=10, steps=6, base=2, dtype=torch.int32)
# It first creates an tensor containing numbers equidistance from each other like `linspace` -> {0, 2, 4, 6, 8, 10} then the log of what numbers that will give us this perticular tensor -> {log2(1), log2(4)...} 

tensor([   1,    4,   16,   64,  256, 1024], dtype=torch.int32)

---

In [42]:
x: torch.Tensor = torch.tensor([[1,2,3],[4,5,6]])
x

tensor([[1, 2, 3],
        [4, 5, 6]])

In [43]:
print(x.shape)
print(x.size(dim=1)) # You can pass the dimension as well → dim = 1 (rows - outermost dimension)

torch.Size([2, 3])
3


numpy/pandas `axis` paremeter and torch `dim` paremater are same → 0 (innermost), 1(second innermost) ...

In [51]:
# Create an uninitialized tensor with the same shape as the input tensor
y: torch.Tensor = torch.empty_like(
    input=x,
    memory_format=torch.preserve_format, # the way how this tensor(y) get stored in memory is exactly same as the tensor that you are using to create(x) → how tensor get stored in the memory
    dtype=torch.float32,
    layout=torch.strided, # dense tensor
    device=device,
    pin_memory=False,
    requires_grad=True
)

print(y)

tensor([[1.5030e+00, 1.7115e-02, 7.6183e-01],
        [2.7644e-01, 1.0026e+00, 3.5559e-04]], device='cuda:0',
       requires_grad=True)


In [52]:
a = torch.rand(size=[5], requires_grad=True)
b = torch.ones_like(input=a, memory_format=torch.preserve_format)
print(b.requires_grad) # no gradient computations by default

print(id(a[1]), id(b[1]))
print(a[1].item(), b[1].item()) 
# temporial behavour of python's id() function for numpy arrays and pytorch tensors explination → /Notes/id method & Torch Memory Format

b[0] = 32
print(a[0].item())

False
127373507940688 127373507940688
0.7265945672988892 1.0
0.3551097512245178


In [53]:
# torch.preserve_format
print(id(x[0][0]))
print(id(y[0][0]))

127374325070416
127374325068624


In [ ]:
# more explaination → /Notes/id method & Torch Memory Format
image: torch.Tensor = torch.rand(size=[10, 3, 5, 5])
print(f"memory format before → {image.stride()}")

image = image.to(dtype=torch.float32, device="cuda", memory_format=torch.channels_last)
"""
    torch.channels_last - NHWC [red_pix_0, green_pix_0, blue_pix_0, red_pix_1...]
    torch.contiguous_format - NCHW [red_pix_0, red_pix_1, .... , green_pix_0, ...]
"""
print(f"memory format after → {image.stride()}")

if image.is_contiguous(memory_format=torch.contiguous_format):
    print("image stored by {batch_size, channels, height, width}")
else:
    print("image stored by {batch_size, height, width, channels}")

memory format before → (75, 25, 5, 1)
memory format after → (75, 1, 15, 3)
image stored by {batch_size, height, width, channels}


In [71]:
tensor = torch.empty(size=[2, 3], dtype=torch.float32, device=device, requires_grad=True)
tensor_like: torch.Tensor = torch.empty_like(input=tensor)

print(f"shape: {tensor.shape, tensor_like.shape}")
print(f"dtype: {tensor.dtype, tensor_like.dtype}")
print(f"device: {tensor.device, tensor_like.device}")

try:
    print(f"gradients: {tensor.requires_grad, tensor_like.required_grad}")
except Exception as e:
    print("tensor_like -", e)

shape: (torch.Size([2, 3]), torch.Size([2, 3]))
dtype: (torch.float32, torch.float32)
device: (device(type='cuda', index=0), device(type='cuda', index=0))
tensor_like - 'Tensor' object has no attribute 'required_grad'


In [56]:
torch.ones_like(x)

tensor([[1, 1, 1],
        [1, 1, 1]])

In [57]:
torch.rand_like(x, dtype=torch.float32)

tensor([[0.7051, 0.4144, 0.8884],
        [0.4462, 0.7413, 0.0865]])

`_like()` These methods create a new tensor with the exact same shape, dtype, and device as an existing input tensor but not `gradient computations`.

---

In [73]:
# find data type
x.dtype

torch.int64

In [74]:
# assign data type
torch.tensor([1.0, 2.0, 3.0], dtype = torch.int32)

tensor([1, 2, 3], dtype=torch.int32)

In [75]:
torch.tensor([1, 2, 3], dtype = torch.float64)

tensor([1., 2., 3.], dtype=torch.float64)

In [76]:
# using to() - type conversion, shifting from CPU → GPU, changing memory format
x.to(torch.float32)

tensor([[1., 2., 3.],
        [4., 5., 6.]])

| **Data Type**             | **Dtype**         | **Description**                                                                                                                                                                |
|---------------------------|-------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **32-bit Floating Point** | `torch.float32`   | Standard floating-point type used for most deep learning tasks. Provides a balance between precision and memory usage.                                                         |
| **64-bit Floating Point** | `torch.float64`   | Double-precision floating point. Useful for high-precision numerical tasks but uses more memory.                                                                               |
| **16-bit Floating Point** | `torch.float16`   | Half-precision floating point. Commonly used in mixed-precision training to reduce memory and computational overhead on modern GPUs.                                            |
| **BFloat16**              | `torch.bfloat16`  | Brain floating-point format with reduced precision compared to `float16`. Used in mixed-precision training, especially on TPUs.                                                |
| **8-bit Floating Point**  | `torch.float8`    | Ultra-low-precision floating point. Used for experimental applications and extreme memory-constrained environments (less common).                                               |
| **8-bit Integer**         | `torch.int8`      | 8-bit signed integer. Used for quantized models to save memory and computation in inference.                                                                                   |
| **16-bit Integer**        | `torch.int16`     | 16-bit signed integer. Useful for special numerical tasks requiring intermediate precision.                                                                                    |
| **32-bit Integer**        | `torch.int32`     | Standard signed integer type. Commonly used for indexing and general-purpose numerical tasks.                                                                                  |
| **64-bit Integer**        | `torch.int64`     | Long integer type. Often used for large indexing arrays or for tasks involving large numbers.                                                                                  |
| **8-bit Unsigned Integer**| `torch.uint8`     | 8-bit unsigned integer. Commonly used for image data (e.g., pixel values between 0 and 255).                                                                                    |
| **Boolean**               | `torch.bool`      | Boolean type, stores `True` or `False` values. Often used for masks in logical operations.                                                                                      |
| **Complex 64**            | `torch.complex64` | Complex number type with 32-bit real and 32-bit imaginary parts. Used for scientific and signal processing tasks.                                                               |
| **Complex 128**           | `torch.complex128`| Complex number type with 64-bit real and 64-bit imaginary parts. Offers higher precision but uses more memory.                                                                 |
| **Quantized Integer**     | `torch.qint8`     | Quantized signed 8-bit integer. Used in quantized models for efficient inference.                                                                                              |
| **Quantized Unsigned Integer** | `torch.quint8` | Quantized unsigned 8-bit integer. Often used for quantized tensors in image-related tasks.                                                                                     |


---

#### Mathematical Operations

In [77]:
x = torch.rand(2,2)
x

tensor([[0.6347, 0.7970],
        [0.3015, 0.8549]])

In [78]:
# Scalar Operations

# addition
print(x + 2, end = "\n\n")

# substraction
print(x - 2, end = "\n\n")

# multiplication
print(x * 3, end = "\n\n")

# division
print(x / 3, end = "\n\n")

# int division
print((x * 100) // 3, end = "\n\n")

# mod
print(((x * 100) // 3) % 2, end = "\n\n")

# power
print(x ** 2)

tensor([[2.6347, 2.7970],
        [2.3015, 2.8549]])

tensor([[-1.3653, -1.2030],
        [-1.6985, -1.1451]])

tensor([[1.9041, 2.3910],
        [0.9045, 2.5647]])

tensor([[0.2116, 0.2657],
        [0.1005, 0.2850]])

tensor([[21., 26.],
        [10., 28.]])

tensor([[1., 0.],
        [0., 0.]])

tensor([[0.4028, 0.6352],
        [0.0909, 0.7309]])


In [80]:
# floor (closest smaller int value) division
(torch.tensor([1.0, 2.0, 3.0]) * 100) // 3

tensor([ 33.,  66., 100.])

In [81]:
# Element wise operation
a = torch.rand(2,3)
b = torch.rand(2,3)

print(a)
print(b)

tensor([[0.3261, 0.0928, 0.9782],
        [0.3096, 0.6806, 0.2401]])
tensor([[0.9080, 0.2860, 0.8329],
        [0.0372, 0.2802, 0.5181]])


In [82]:
# Shape must be same or able to broadcast

# add
print(a + b, end="\n\n")

# sub
print(a - b, end="\n\n")

# multiply
print(a * b, end="\n\n")

# division
print(a / b, end="\n\n")

# power
print(a ** b, end="\n\n")

# mod
print(a % b)

tensor([[1.2341, 0.3788, 1.8110],
        [0.3469, 0.9608, 0.7582]])

tensor([[-0.5818, -0.1931,  0.1453],
        [ 0.2724,  0.4004, -0.2780]])

tensor([[0.2961, 0.0265, 0.8147],
        [0.0115, 0.1907, 0.1244]])

tensor([[0.3592, 0.3247, 1.1744],
        [8.3185, 2.4290, 0.4634]])

tensor([[0.3615, 0.5068, 0.9818],
        [0.9573, 0.8978, 0.4775]])

tensor([[0.3261, 0.0928, 0.1453],
        [0.0119, 0.1202, 0.2401]])


In [83]:
c = torch.tensor([1, -2, 3, -4])
c

tensor([ 1, -2,  3, -4])

In [84]:
# abs
torch.abs(c)

tensor([1, 2, 3, 4])

In [85]:
# negative
torch.neg(c)

tensor([-1,  2, -3,  4])

In [86]:
# Inplace operations
print(torch.abs_(c))
print(c)

print(torch.neg_(c))
print(c)

tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4])
tensor([-1, -2, -3, -4])
tensor([-1, -2, -3, -4])


In [87]:
c.fill_(value=10)
# In-place equivalent → c = torch.full_like(c, fill_value=10)

tensor([10, 10, 10, 10])

In [95]:
x = torch.empty_like(input=c)

c.copy_(other=x) # c gets updated, In-place equivalent → c = x.clone()
cloned_c: torch.Tensor = x.clone()

print((c == cloned_c).all().item())

True


In [97]:
d: torch.Tensor = torch.tensor([1.9, 2.3, 3.7, 4.4, 6.5])
d

tensor([1.9000, 2.3000, 3.7000, 4.4000, 6.5000])

In [98]:
# round
torch.round(d) # 6.5 -> 6 (get floored not ceiled)

tensor([2., 2., 4., 4., 6.])

In [99]:
# ceil - nearest bigger integer value
torch.ceil(d)

tensor([2., 3., 4., 5., 7.])

In [100]:
# floor - nearest smaller integer value
torch.floor(d)

tensor([1., 2., 3., 4., 6.])

In [101]:
# clamp | clip - Clipping the values in a certing range
torch.clip(d, min = -1, max = 1)
torch.clamp(d, min = -1, max = 1)

tensor([1., 1., 1., 1., 1.])

In [102]:
e: torch.Tensor = torch.randint(size=(2,3), low=0, high=10, dtype=torch.float32)
e

tensor([[3., 9., 4.],
        [1., 0., 1.]])

In [ ]:
# NumPy `axis` vs Pandas `axis` vs PyTorch `dim` → All are same or might change with function
import numpy as np
import pandas as pd
import torch

print(np.array(
    [[1, 2, 3], [4, 5, 6]]
).sum(axis=0))

print(pd.DataFrame(
    [[1, 2, 3], [4, 5, 6]]
).sum(axis=0).to_numpy()) # string arguments → `index` = 0 and `columns` = 1

print(torch.tensor(
    [[1, 2, 3], [4, 5, 6]]
).sum(dim=0))

[5 7 9]
[5 7 9]
tensor([5, 7, 9])


In [104]:
# Aggregate operation

# sum
print(torch.sum(e))

# sum along columns
print(torch.sum(e, dim = 0, keepdim=False)) # same as `axis` parameter in numpy

# sum along rows
print(torch.sum(e, dim = 1, keepdim=False))

# keepdim = True
print(torch.sum(e, dim = 1, keepdim=True))

tensor(18.)
tensor([4., 9., 5.])
tensor([16.,  2.])
tensor([[16.],
        [ 2.]])


If `keepdim` is `False`, the output tensor will not retain the dimension across which the sum was performed.

In [105]:
# mean
torch.mean(e)

# mean along col
torch.mean(e, dim=0)

tensor([2.0000, 4.5000, 2.5000])

In [106]:
# median
torch.median(e)

tensor(1.)

In [107]:
# max and min
torch.max(e)
torch.min(e)

tensor(0.)

In [108]:
# product
torch.prod(e) # .mul() is a element-wise multiplication between two matrics

tensor(0.)

In [109]:
# standard deviation
torch.std(e)

tensor(3.2863)

In [110]:
# variance
torch.var(e)

tensor(10.8000)

In [111]:
# argmax
torch.argmax(e)

tensor(1)

In [112]:
# argmin
torch.argmin(e)

tensor(4)

In [113]:
# More _methods()
a = torch.full_like(input=e, fill_value=10)
b = a.clone().copy_(a) * 100 # the shape of a and b must be same for copy_ method
print(a, b, sep="\n")

# clone and copy_ → both the tensor elements are pointing to the same memory location

a.clone().add_(b) # a = a + b (ignore clone)
a.clone().sub_(b) # a = a - b 
a.clone().mul_(b) # a = a * b (element wise)

a.zero_() # x = torch.zeros_like(x)

tensor([[10., 10., 10.],
        [10., 10., 10.]])
tensor([[1000., 1000., 1000.],
        [1000., 1000., 1000.]])


tensor([[0., 0., 0.],
        [0., 0., 0.]])

---

#### Matrix operations

In [114]:
f: torch.Tensor = torch.randint(size=(2,3), low=0, high=10)
g: torch.Tensor = torch.randint(size=(3,2), low=0, high=10)

print(f)
print(g)

tensor([[6, 0, 8],
        [3, 8, 5]])
tensor([[8, 3],
        [0, 8],
        [5, 8]])


In [115]:
# matrix multiplcation - dot product
torch.matmul(f, g)

tensor([[ 88,  82],
        [ 49, 113]])

In [116]:
vector1: torch.Tensor = torch.tensor([1, 2])
vector2: torch.Tensor = torch.tensor([3, 4])

# dot product - only works with Vectors
torch.dot(vector1, vector2)

tensor(11)

In [117]:
# transpose
torch.transpose(input=f, dim0=0, dim1=1)

tensor([[6, 3],
        [0, 8],
        [8, 5]])

In [118]:
torch.transpose(f, dim0 = 1, dim1 = 0) # Transpose is same as `reshape` operation by swapping the dimensions

tensor([[6, 3],
        [0, 8],
        [8, 5]])

In [119]:
# 0th axis - 1, 1st axis - 2, 2nd axis - 3 → consider size as a list and axis is an index
torch.rand(size=[1, 2, 3,]).transpose(dim0=0, dim1=2).shape

torch.Size([3, 2, 1])

In [120]:
# Implace Transpose → only works with 2D
f.t_()

tensor([[6, 3],
        [0, 8],
        [8, 5]])

In [123]:
temp: torch.Tensor = torch.tensor([
    [[1], [2]],
    [[1], [3]],
    [[1], [4]],
])

print(temp.shape)
print(torch.transpose(temp, dim0=0, dim1=1).shape) # Check the dimensions and its index correctly

torch.Size([3, 2, 1])
torch.Size([2, 3, 1])


Tip: To visualize a transpose, flatten the tensor into a 1D array and then re-arrange the elements according to the target shape.
<br>For a tensor with shape [2, 3, 1], the 0th dimension has size 2, the 1st dimension has size 3, and so on.

In [124]:
# Convert any nD tensor to 1D tensor
print(torch.tensor([[1, 2], [3, 4]]).ravel())
print(torch.tensor([[1, 2], [3, 4]]).flatten())
print(torch.tensor([[[1], [2]], [[3], [4]]]).reshape(-1))

tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4])


In [125]:
h: torch.Tensor = torch.randint(size=(3,3), low=0, high=10, dtype=torch.float32)
h

tensor([[3., 0., 2.],
        [1., 8., 9.],
        [5., 5., 0.]])

In [126]:
# determinant
torch.det(h)

tensor(-205.)

In [127]:
# inverse
torch.inverse(h)

tensor([[ 0.2195, -0.0488,  0.0780],
        [-0.2195,  0.0488,  0.1220],
        [ 0.1707,  0.0732, -0.1171]])

---

#### Comparison operations

In [128]:
i: torch.Tensor = torch.randint(size=(2,3), low=0, high=10)
j: torch.Tensor = torch.randint(size=(2,3), low=0, high=10)

print(i)
print(j)

tensor([[5, 1, 4],
        [7, 6, 7]])
tensor([[9, 4, 0],
        [8, 8, 0]])


In [129]:
# Element wise comparison - Shape must be same or able to broadcast

# greater than
print(i > j)

# less than
print(i < j)

# equal to
print(i == j)

# not equal to
print(i != j)

tensor([[False, False,  True],
        [False, False,  True]])
tensor([[ True,  True, False],
        [ True,  True, False]])
tensor([[False, False, False],
        [False, False, False]])
tensor([[True, True, True],
        [True, True, True]])


In [130]:
# Boolean Indexing - Reshaped into 1D
i[i > j]

tensor([4, 7])

---

#### Special functions

In [131]:
k: torch.Tensor = torch.randint(size=(2,3), low=0, high=10, dtype=torch.float64)
k

tensor([[0., 8., 3.],
        [9., 2., 4.]], dtype=torch.float64)

In [132]:
# log - base e
torch.log(input = k)

# log - base 10
torch.log10(input = k)

tensor([[  -inf, 0.9031, 0.4771],
        [0.9542, 0.3010, 0.6021]], dtype=torch.float64)

In [134]:
# Checking the size of the elements in the tensor
tensor = torch.log10(input = k) # implicit type conversion from float32 to float 64
print(tensor.dtype)
print(tensor.element_size())  # size in bytes of a single element

torch.float64
8


In [135]:
# exp
torch.exp(k)

tensor([[1.0000e+00, 2.9810e+03, 2.0086e+01],
        [8.1031e+03, 7.3891e+00, 5.4598e+01]], dtype=torch.float64)

In [136]:
# sqrt
torch.sqrt(k)

tensor([[0.0000, 2.8284, 1.7321],
        [3.0000, 1.4142, 2.0000]], dtype=torch.float64)

In [137]:
# sigmoid
torch.sigmoid(k)

tensor([[0.5000, 0.9997, 0.9526],
        [0.9999, 0.8808, 0.9820]], dtype=torch.float64)

In [138]:
# softmax
torch.softmax(k, dim = 1)

tensor([[3.3311e-04, 9.9298e-01, 6.6906e-03],
        [9.9241e-01, 9.0496e-04, 6.6868e-03]], dtype=torch.float64)

In [139]:
# relu
torch.relu(k)

tensor([[0., 8., 3.],
        [9., 2., 4.]], dtype=torch.float64)

---

#### Inplace Operations

In [140]:
m: torch.Tensor = torch.rand(2,3)
n: torch.Tensor = torch.rand(2,3)

print(m)
print(n)

tensor([[0.0952, 0.4097, 0.7290],
        [0.7225, 0.6214, 0.5254]])
tensor([[0.6997, 0.4187, 0.3943],
        [0.4495, 0.7858, 0.0359]])


In [141]:
# _ methods
m.add_(n) # It returns the new matrix

tensor([[0.7949, 0.8284, 1.1233],
        [1.1719, 1.4073, 0.5612]])

In [142]:
m # Updated

tensor([[0.7949, 0.8284, 1.1233],
        [1.1719, 1.4073, 0.5612]])

In [143]:
n

tensor([[0.6997, 0.4187, 0.3943],
        [0.4495, 0.7858, 0.0359]])

In [144]:
torch.relu(m)

tensor([[0.7949, 0.8284, 1.1233],
        [1.1719, 1.4073, 0.5612]])

In [145]:
m.relu_()

tensor([[0.7949, 0.8284, 1.1233],
        [1.1719, 1.4073, 0.5612]])

In [146]:
m

tensor([[0.7949, 0.8284, 1.1233],
        [1.1719, 1.4073, 0.5612]])

---

#### Copying a Tensor

In [147]:
a = torch.tensor(
    data = [[1,2,3], [4,5,6]]
)
b = a.clone()

print(a, b, sep = '\n')

print(id(a), id(b))
print(id(a[0,0]), id(b[0][0]))

tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([[1, 2, 3],
        [4, 5, 6]])
127372819695952 127372819701072
127372819697104 127372819706576


In [148]:
a[0][0] = 10

print(a, b, sep = '\n')

tensor([[10,  2,  3],
        [ 4,  5,  6]])
tensor([[1, 2, 3],
        [4, 5, 6]])


---

A third-order polynomial, trained to predict `y = sin(x) from -pi to pi` by minimizing squared Euclidean distance.

In [152]:
# Variables Initialization
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype: torch._C.dtype = torch.float
learning_rate = 1e-6

# Dataset
x = torch.linspace(start=-torch.pi, end=torch.pi, steps=2000) # shape -> 2000 1D
y = torch.sin(input=x) # Shape -> 2000 1D

# Random Weights → for 3 degree term | shape -> Scaler
a = torch.randn((), dtype=dtype)
b = torch.randn((), dtype=dtype)
c = torch.randn((), dtype=dtype)
d = torch.randn((), dtype=dtype)

def y_pred_equation(X_train):
    y_pred: torch.Tensor = a + (b * x) + (c * x.pow(2)) + (d * x.pow(3))
    return y_pred # shape -> 2000 1D

def main():
    # Training
    for i in range(2500):
        # Prediction
        y_pred = y_pred_equation(x)

        # Loss Calculation
        loss = (y - y_pred).pow(2).sum().item()

        if i % 100 == 99:
            print(f"loss at {i + 1}it epoch → {loss}")

        # Gradients Calculation
        grad_y_pred = -2 * (y - y_pred) # shape -> 2000
        grad_a: torch.Tensor = grad_y_pred.sum()
        grad_b: torch.Tensor = torch.matmul(input = grad_y_pred, other = x).sum()
        grad_c: torch.Tensor = torch.matmul(input = grad_y_pred, other = x.pow(2)).sum()
        grad_d: torch.Tensor = torch.matmul(input = grad_y_pred, other = x.pow(3)).sum()

        # Updating
        global a, b, c, d
        a = a - learning_rate * grad_a
        b -= learning_rate * grad_b
        c -= learning_rate * grad_c
        d -= learning_rate * grad_d

main()

loss at 100it epoch → 2775.90185546875
loss at 200it epoch → 1865.41796875
loss at 300it epoch → 1255.5062255859375
loss at 400it epoch → 846.6444091796875
loss at 500it epoch → 572.35107421875
loss at 600it epoch → 388.19110107421875
loss at 700it epoch → 264.4458923339844
loss at 800it epoch → 181.22547912597656
loss at 900it epoch → 125.20973205566406
loss at 1000it epoch → 87.4713134765625
loss at 1100it epoch → 62.022823333740234
loss at 1200it epoch → 44.84558868408203
loss at 1300it epoch → 33.239620208740234
loss at 1400it epoch → 25.390172958374023
loss at 1500it epoch → 20.07593536376953
loss at 1600it epoch → 16.47416877746582
loss at 1700it epoch → 14.030486106872559
loss at 1800it epoch → 12.370711326599121
loss at 1900it epoch → 11.24215316772461
loss at 2000it epoch → 10.473901748657227
loss at 2100it epoch → 9.950310707092285
loss at 2200it epoch → 9.59310531616211
loss at 2300it epoch → 9.349111557006836
loss at 2400it epoch → 9.18226146697998
loss at 2500it epoch → 9.